# SAR Slick Lab
Five-class oil-spill segmentation with U-Net and DeepLabV3+.
Attach the `nabilsherif/oil-spill` dataset and select a GPU accelerator. Enable internet to fetch the torchvision ResNet-18 ImageNet weights. The cells below write the exact same modules as the local repo. The final run opens the official test split only after all four validation-selected models are trained.

In [ ]:
import os
import sys
from pathlib import Path

import torch
assert torch.cuda.is_available(), "Select a GPU accelerator in Kaggle settings."
work = Path("/kaggle/working/sar-slick-lab")
work.mkdir(parents=True, exist_ok=True)
os.chdir(work)
dataset = Path("/kaggle/input/oil-spill")
if not dataset.is_dir():
    dataset = Path("/kaggle/input/datasets/nabilsherif/oil-spill")
assert dataset.is_dir(), "Attach the nabilsherif/oil-spill dataset before running."
print(torch.cuda.get_device_name(), dataset)

In [ ]:
%%writefile data_io.py
"""Pair official splits, audit exact labels, and prepare training tensors."""
from pathlib import Path

import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset

NAMES = ("Sea", "Oil spill", "Look-alike", "Ship", "Land")
COLORS = np.array([(0, 0, 0), (0, 255, 255), (255, 0, 0), (153, 76, 0), (0, 153, 0)], dtype=np.uint8)
MEAN = torch.tensor([.485, .456, .406])[:, None, None]
STD = torch.tensor([.229, .224, .225])[:, None, None]
EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def split_pairs(root, split):
    """Find exactly one split folder containing matching image and mask folders."""
    candidates = []
    for folder in [root, *root.rglob("*")]:
        if not folder.is_dir() or folder.name.lower() not in {split, split + "ing"}:
            continue
        subdirs = {p.name.lower(): p for p in folder.iterdir() if p.is_dir()}
        images = next((subdirs[key] for key in ("images", "image", "imgs") if key in subdirs), None)
        masks = next((subdirs[key] for key in ("labels", "label", "masks", "mask", "annotations") if key in subdirs), None)
        if images is not None and masks is not None:
            candidates.append((images, masks))
    if len(candidates) != 1:
        raise ValueError(f"Expected one {split}/images + {split}/labels directory pair under {root}; found {candidates}")

    def indexed(folder):
        files = {}
        for path in sorted(folder.iterdir()):
            if path.suffix.lower() in EXTENSIONS and path.is_file():
                key = path.stem.casefold()
                if key in files:
                    raise ValueError(f"Duplicate stem: {path}")
                files[key] = path
        return files

    images, masks = map(indexed, candidates[0])
    if not images or images.keys() != masks.keys():
        raise ValueError(f"Unmatched {split} stems: {sorted(images.keys() ^ masks.keys())[:10]}")
    return [(images[key], masks[key]) for key in sorted(images)]


def decode(mask):
    """Accept real ID masks or decode palette/RGB masks by exact color."""
    raw = np.asarray(mask)
    if mask.mode != "P" and raw.ndim == 2 and raw.min() >= 0 and raw.max() < len(NAMES):
        return raw.astype(np.uint8)
    rgb = np.asarray(mask.convert("RGB"))
    ids = np.full(rgb.shape[:2], 255, dtype=np.uint8)
    for index, color in enumerate(COLORS):
        ids[np.all(rgb == color, axis=-1)] = index
    if np.any(ids == 255):
        unexpected = np.unique(rgb[ids == 255], axis=0)
        raise ValueError(f"Unknown mask colors: {unexpected[:10].tolist()}")
    return ids


def read_pair(pair):
    image_path, mask_path = pair
    with Image.open(image_path) as source, Image.open(mask_path) as mask:
        if source.size != mask.size:
            raise ValueError(f"Image and mask dimensions disagree: {image_path}, {mask_path}")
        try:
            return source.convert("RGB"), decode(mask)
        except ValueError as error:
            raise ValueError(f"{mask_path}: {error}") from error


def audit(pairs):
    counts = np.zeros(len(NAMES), dtype=np.int64)
    dimensions = {}
    for pair in pairs:
        image, ids = read_pair(pair)
        counts += np.bincount(ids.ravel(), minlength=len(NAMES))
        key = f"{image.width}x{image.height}"
        dimensions[key] = dimensions.get(key, 0) + 1
    return {"images": len(pairs), "dimensions": dimensions, "pixel_counts": counts.tolist()}


class Tiles(Dataset):
    def __init__(self, pairs, size, augment=False):
        self.pairs = pairs
        self.size = (size, size)
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image, ids = read_pair(self.pairs[index])
        image = np.asarray(image.resize(self.size, Image.Resampling.BILINEAR))
        ids = np.asarray(Image.fromarray(ids).resize(self.size, Image.Resampling.NEAREST))
        if self.augment:
            if torch.rand(()) < .5:
                image, ids = np.flip(image, axis=1), np.flip(ids, axis=1)
            turns = int(torch.randint(0, 4, ()).item())
            image, ids = np.rot90(image, turns), np.rot90(ids, turns)
        x = torch.from_numpy(image.copy()).permute(2, 0, 1).float() / 255
        y = torch.from_numpy(ids.copy()).long()
        return (x - MEAN) / STD, y, self.pairs[index][0].stem


In [ ]:
%%writefile models.py
"""Scratch U-Net and DeepLabV3+ with a torchvision ResNet-18 encoder."""
import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models import resnet18, ResNet18_Weights


def conv(cin, cout, kernel=3, dilation=1, groups=1):
    return nn.Sequential(
        nn.Conv2d(cin, cout, kernel, padding=(kernel // 2) * dilation,
                  dilation=dilation, groups=groups, bias=False),
        nn.GroupNorm(8, cout),
        nn.ReLU(inplace=True),
    )


def double_conv(cin, cout):
    return nn.Sequential(conv(cin, cout), conv(cout, cout))


def upsample(tensor, shape):
    return F.interpolate(tensor, size=shape, mode="bilinear", align_corners=False)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        widths = (24, 48, 96, 192)
        self.down = nn.ModuleList(double_conv(a, b) for a, b in zip((3, *widths), widths))
        self.pool = nn.MaxPool2d(2)
        self.middle = double_conv(192, 384)
        self.up = nn.ModuleList(double_conv(a + b, b) for a, b in zip((384, 192, 96, 48), reversed(widths)))
        self.head = nn.Conv2d(24, 5, 1)

    def forward(self, x):
        skips = []
        for block in self.down:
            x = block(x)
            skips.append(x)
            x = self.pool(x)
        x = self.middle(x)
        for block, skip in zip(self.up, reversed(skips)):
            x = block(torch.cat([upsample(x, skip.shape[-2:]), skip], dim=1))
        return self.head(x)


def separable(cin, cout, dilation=1):
    return nn.Sequential(conv(cin, cin, dilation=dilation, groups=cin), conv(cin, cout, 1))


class DeepLabV3Plus(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.encoder = resnet18(weights=weights)
        del self.encoder.fc, self.encoder.avgpool
        # Keep layer4 on the layer3 grid and enlarge its receptive field.
        layer4 = self.encoder.layer4
        layer4[0].conv1.stride = (1, 1)
        layer4[0].downsample[0].stride = (1, 1)
        for layer in (layer4[0].conv2, layer4[1].conv1, layer4[1].conv2):
            layer.dilation = layer.padding = (2, 2)
        self.aspp = nn.ModuleList([conv(512, 96, 1), *(separable(512, 96, rate) for rate in (3, 6, 9))])
        self.global_context = nn.Sequential(nn.AdaptiveAvgPool2d(1), conv(512, 96, 1))
        self.project = nn.Sequential(conv(480, 96, 1), nn.Dropout(.1))
        self.low = conv(64, 32, 1)
        self.decoder = nn.Sequential(separable(128, 96), separable(96, 96), nn.Conv2d(96, 5, 1))

    def forward(self, x):
        encoder = self.encoder
        stem = encoder.relu(encoder.bn1(encoder.conv1(x)))
        low = encoder.layer1(encoder.maxpool(stem))
        high = encoder.layer4(encoder.layer3(encoder.layer2(low)))
        branches = [branch(high) for branch in self.aspp]
        branches.append(upsample(self.global_context(high), high.shape[-2:]))
        context = upsample(self.project(torch.cat(branches, dim=1)), low.shape[-2:])
        return upsample(self.decoder(torch.cat([context, self.low(low)], dim=1)), x.shape[-2:])


In [ ]:
%%writefile metrics.py
"""Pixel-level loss and metrics for mutually exclusive segmentation classes."""
import torch
from torch.nn import functional as F


def objective(logits, target, weights=None):
    cross_entropy = F.cross_entropy(logits, target, weight=weights)
    if weights is None:
        return cross_entropy
    probabilities = logits.float().softmax(dim=1)
    truth = F.one_hot(target, 5).movedim(-1, 1).float()
    dimensions = (0, 2, 3)
    intersection = (probabilities * truth).sum(dimensions)
    support = truth.sum(dimensions)
    overlap = (2 * intersection + 1e-6) / (probabilities.sum(dimensions) + support + 1e-6)
    return cross_entropy + .5 * (1 - overlap[support > 0].mean())


def confusion(predicted, target):
    return torch.bincount((5 * target + predicted).reshape(-1), minlength=25).reshape(5, 5)


def scores(matrix):
    matrix = matrix.double()
    correct = matrix.diag()
    actual = matrix.sum(1)
    predicted = matrix.sum(0)
    union = actual + predicted - correct
    iou = [float(a / b) if b else None for a, b in zip(correct, union)]
    dice = [float(2 * a / b) if b else None for a, b in zip(correct, actual + predicted)]
    return {
        "miou": sum(value for value in iou if value is not None) / int((union > 0).sum()),
        "iou": iou, "dice": dice, "accuracy": float(correct.sum() / matrix.sum()),
        "confusion": matrix.long().tolist(),
    }


In [ ]:
%%writefile run.py
"""Run the four controlled experiments and evaluate validation-selected weights."""
import argparse
import json
import random
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from data_io import COLORS, MEAN, NAMES, STD, Tiles, audit, split_pairs
from metrics import confusion, objective, scores
from models import DeepLabV3Plus, UNet

EXPERIMENTS = {"unet-ce": ("unet", False), "unet-balanced": ("unet", True),
               "deeplab-ce": ("deeplab", False), "deeplab-balanced": ("deeplab", True)}


def write_json(path, data):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(data, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)


def save_checkpoint(path, data):
    temp = path.with_suffix(".tmp")
    torch.save(data, temp)
    temp.replace(path)


def seed(value):
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)


def loader(pairs, args, training=False, epoch=0):
    return DataLoader(Tiles(pairs, args.size, training), batch_size=args.batch,
                      shuffle=training, num_workers=args.workers,
                      generator=torch.Generator().manual_seed(args.seed + epoch),
                      pin_memory=args.device == "cuda")


def visualize(panels, path):
    fig, axes = plt.subplots(len(panels), len(panels[0][1]), squeeze=False,
                             figsize=(4 * len(panels[0][1]), 3.2 * len(panels)))
    for row, (name, images) in zip(axes, panels):
        for ax, (title, image) in zip(row, images):
            ax.imshow(image, interpolation="nearest")
            ax.set_title(f"{name} | {title}", fontsize=9)
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=140)
    plt.close(fig)


def image_for_display(image):
    return ((image.cpu() * STD + MEAN).clamp(0, 1)).permute(1, 2, 0).numpy()


def pass_epoch(model, batches, device, weights, optimizer=None, scaler=None, collect=False):
    model.train(optimizer is not None)
    total_loss, total_images = 0., 0
    matrix = torch.zeros((5, 5), dtype=torch.int64)
    selected = {}
    for x, y, names in batches:
        x, y = x.to(device), y.to(device)
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(optimizer is not None):
            with torch.autocast(device, enabled=device == "cuda"):
                logits = model(x)
                loss = objective(logits, y, weights)
            if optimizer is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        predictions = logits.detach().argmax(dim=1)
        matrix += confusion(predictions, y).cpu()
        total_loss += float(loss.detach()) * len(x)
        total_images += len(x)
        if collect:
            for image, target, pred, name in zip(x.cpu(), y.cpu().numpy(), predictions.cpu().numpy(), names):
                oil_errors = np.count_nonzero(((target == 1) & (pred == 2)) | ((target == 2) & (pred == 1)))
                ship_misses = np.count_nonzero((target == 3) & (pred != 3))
                for kind, count in [("First test tile", 1), ("Oil/look-alike errors", oil_errors), ("Missed ship pixels", ship_misses)]:
                    if count > selected.get(kind, (0, None))[0]:
                        selected[kind] = (count, (name, [("SAR", image_for_display(image)),
                                                        ("Ground truth", COLORS[target]), ("Prediction", COLORS[pred])]))
    return {"loss": total_loss / total_images, **scores(matrix)}, selected


def make_model(architecture, pretrained):
    return UNet() if architecture == "unet" else DeepLabV3Plus(pretrained)


def plot_history(history, path):
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
    for ax, metric, label in zip(axes, ("loss", "miou"), ("Objective loss", "Mean IoU")):
        for split in ("train", "validation"):
            ax.plot([row["epoch"] for row in history], [row[split][metric] for row in history],
                    marker="o", markersize=3, label=split)
        ax.set(xlabel="Epoch", ylabel=label)
        ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=140)
    plt.close(fig)


def plot_confusion(matrix, path):
    values = np.asarray(matrix)
    normalized = values / np.maximum(values.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(normalized, vmin=0, vmax=1, cmap="Blues")
    ax.set(xticks=range(5), yticks=range(5), xticklabels=NAMES, yticklabels=NAMES,
           xlabel="Predicted class", ylabel="True class")
    plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
    for i, j in np.ndindex(5, 5):
        ax.text(j, i, f"{normalized[i, j]:.2f}", ha="center", va="center",
                color="white" if normalized[i, j] > .55 else "black")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=140)
    plt.close(fig)


def train(args, name, train_pairs, validation_pairs, weights, split_manifest):
    architecture, balanced = EXPERIMENTS[name]
    weights = weights.to(args.device) if balanced else None
    directory = args.out / name
    directory.mkdir(exist_ok=True)
    latest = directory / "latest.pt"
    checkpoint = torch.load(latest, map_location="cpu", weights_only=True) if latest.exists() else None
    config = {"architecture": architecture, "balanced": balanced,
              "encoder_weights": "ImageNet-1K V1" if architecture == "deeplab" and not args.smoke else None,
              "size": args.size, "batch": args.batch, "seed": args.seed, "lr": args.lr,
              "device": args.device, "smoke": args.smoke, "split": split_manifest,
              "class_weights": weights.tolist() if balanced else None}
    if checkpoint and {**checkpoint["config"], "device": args.device} != config:
        raise ValueError(f"Configuration changed for {name}. Use a new output directory.")
    seed(args.seed)
    model = make_model(architecture, config["encoder_weights"] is not None and checkpoint is None).to(args.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=args.device == "cuda")
    history, best_score, best_epoch = [], -1., 0
    if checkpoint:
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        if checkpoint["config"]["device"] == args.device and checkpoint["scaler"]:
            scaler.load_state_dict(checkpoint["scaler"])
        history, best_score, best_epoch = (checkpoint[key] for key in ("history", "best_score", "best_epoch"))
    if len(history) > args.epochs:
        raise ValueError("The output already contains more epochs than requested.")
    validation = loader(validation_pairs, args)
    for epoch in range(len(history) + 1, args.epochs + 1):
        start = time.perf_counter()
        seed(args.seed + epoch)
        tr, _ = pass_epoch(model, loader(train_pairs, args, True, epoch), args.device, weights, optimizer, scaler)
        val, _ = pass_epoch(model, validation, args.device, weights)
        history.append({"epoch": epoch, "device": args.device, "train": tr, "validation": val, "seconds": time.perf_counter() - start})
        if val["miou"] > best_score:
            best_score, best_epoch = val["miou"], epoch
            save_checkpoint(directory / "best.pt", {"model": model.state_dict(), "config": config, "epoch": epoch})
        save_checkpoint(latest, {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                                 "scaler": scaler.state_dict(), "history": history, "config": config,
                                 "best_score": best_score, "best_epoch": best_epoch})
        write_json(directory / "history.json", history)
        print(f"{name} epoch {epoch}: train mIoU={tr['miou']:.3f} val mIoU={val['miou']:.3f}", flush=True)
    write_json(directory / "history.json", history)
    write_json(directory / "config.json", {**config, "epochs": args.epochs})
    plot_history(history, directory / "history.png")
    return {"experiment": name, "best_epoch": best_epoch, "val_miou": best_score,
            "seconds": sum(row["seconds"] for row in history)}


def resized_weights(pairs, size):
    counts = np.zeros(5, dtype=np.int64)
    for _, mask, _ in Tiles(pairs, size):
        counts += np.bincount(mask.numpy().ravel(), minlength=5)
    if (counts == 0).any():
        raise ValueError(f"Every training class must survive resizing. Counts: {counts.tolist()}")
    weights = np.sqrt(counts.max() / counts).clip(1, 8)
    return torch.tensor(weights, dtype=torch.float32), counts.tolist()


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, required=True)
    parser.add_argument("--out", type=Path, default=Path("runs/final"))
    parser.add_argument("--epochs", type=int, default=16)
    parser.add_argument("--batch", type=int, default=4)
    parser.add_argument("--size", type=int, default=256)
    parser.add_argument("--workers", type=int, default=0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--lr", type=float, default=.001)
    parser.add_argument("--device", choices=["cpu", "cuda"], default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--smoke", action="store_true", help="Run synthetic fixture without encoder weights.")
    parser.add_argument("--train-only", action="store_true", help="Stop before opening the official test set.")
    parser.add_argument("--experiment", choices=EXPERIMENTS, action="append",
                        help="Select training runs. Requires --train-only.")
    args = parser.parse_args()
    if args.experiment and not args.train_only:
        parser.error("--experiment requires --train-only.")
    if args.epochs < 1 or args.batch < 1 or args.size < 32 or args.size % 16:
        parser.error("Use positive epochs and batch size, and an image size divisible by 16 and at least 32.")
    if args.device == "cpu":
        torch.set_num_threads(min(4, torch.get_num_threads()))
    if args.device == "cuda" and not torch.cuda.is_available():
        parser.error("CUDA is unavailable in this PyTorch environment.")
    if args.smoke != (args.data / "SYNTHETIC_FIXTURE").is_file():
        parser.error("Synthetic data must be explicitly marked and run with --smoke.")
    args.out.mkdir(parents=True, exist_ok=True)
    official = split_pairs(args.data, "train")
    if not args.smoke and len(official) != 1002:
        raise ValueError(f"Expected 1002 official training pairs, found {len(official)}.")
    permutation = np.random.default_rng(args.seed).permutation(len(official))
    n_val = max(1, round(.15 * len(official)))
    validation_pairs = [official[i] for i in permutation[:n_val]]
    train_pairs = [official[i] for i in permutation[n_val:]]
    split_manifest = {"train": [p[0].stem for p in train_pairs], "validation": [p[0].stem for p in validation_pairs]}
    weights, counts = resized_weights(train_pairs, args.size)
    write_json(args.out / "audit.json", {"train": audit(train_pairs), "validation": audit(validation_pairs),
                                       "resized_train_counts": counts, "class_names": NAMES, "rgb": COLORS.tolist()})
    samples = []
    for x, y, name in list(Tiles(validation_pairs[:4], args.size)):
        samples.append((name, [("SAR", image_for_display(x)), ("Ground truth", COLORS[y.numpy()])]))
    visualize(samples, args.out / "dataset-preview.png")
    rows = [train(args, name, train_pairs, validation_pairs, weights, split_manifest) for name in dict.fromkeys(args.experiment or EXPERIMENTS)]
    if args.train_only:
        return
    test = split_pairs(args.data, "test")
    if not args.smoke and len(test) != 110:
        raise ValueError(f"Expected 110 official test pairs, found {len(test)}.")
    write_json(args.out / "test-audit.json", audit(test))
    for row in rows:
        name = row["experiment"]
        directory = args.out / name
        checkpoint = torch.load(directory / "best.pt", map_location="cpu", weights_only=True)
        model = make_model(checkpoint["config"]["architecture"], False).to(args.device)
        model.load_state_dict(checkpoint["model"])
        metrics, selected = pass_epoch(model, loader(test, args), args.device, None, collect=True)
        del metrics["loss"]
        row.update(metrics)
        plot_confusion(metrics["confusion"], directory / "confusion.png")
        visualize([panel for _, panel in selected.values()], directory / "predictions.png")
        row["examples"] = {kind: {"tile": panel[0], "pixels": int(count) if kind != "First test tile" else None}
                           for kind, (count, panel) in selected.items()}
    write_json(args.out / "results.json", {"synthetic": args.smoke, "test_images": len(test),
                                         "device": torch.cuda.get_device_name() if args.device == "cuda" else "CPU",
                                         "experiments": rows})
    print(f"Results written to {args.out / 'results.json'}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile report.py
"""Generate the methodology report and add benchmark evidence when available."""
import argparse
import json
from pathlib import Path
from xml.sax.saxutils import escape

from PIL import Image
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import Image as Figure, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

TEAL = colors.HexColor("#06696c")
INK = colors.HexColor("#19303b")
EXPERIMENTS = [("U-Net CE", "Scratch U-Net", "Cross-entropy"),
               ("U-Net balanced", "Scratch U-Net", "Weighted CE + 0.5 Dice loss"),
               ("DeepLab CE", "ImageNet ResNet-18", "Cross-entropy"),
               ("DeepLab balanced", "ImageNet ResNet-18", "Weighted CE + 0.5 Dice loss")]


def load(path):
    return json.loads(path.read_text(encoding="utf-8"))


def format_score(value):
    return "-" if value is None else f"{100 * value:.2f}%"


def build(run, destination):
    result_path = run / "results.json"
    results = load(result_path) if result_path.exists() else None
    synthetic = results is not None and results["synthetic"]
    benchmark = results is not None and not synthetic
    styles = getSampleStyleSheet()
    styles["Title"].fontSize, styles["Title"].leading = 30, 34
    styles["Title"].textColor = INK
    styles["Heading2"].textColor = TEAL
    styles["BodyText"].leading = 12
    story = []

    def paragraph(text, style="BodyText"):
        story.append(Paragraph(escape(text), styles[style]))

    def section(title, text):
        paragraph(title, "Heading2")
        paragraph(text)

    def table(rows, widths):
        grid = Table(rows, colWidths=[width * mm for width in widths], repeatRows=1, hAlign="LEFT")
        grid.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), TEAL), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"), ("FONTSIZE", (0, 0), (-1, -1), 8.3),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.HexColor("#edf5f5"), colors.white]),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 4), ("TOPPADDING", (0, 0), (-1, -1), 4),
        ]))
        story.extend([Spacer(1, 5), grid, Spacer(1, 6)])

    def figure(path, max_height):
        with Image.open(path) as image:
            width, height = image.size
        scale = min(158 * mm / width, max_height * mm / height)
        story.append(Figure(str(path), width=width * scale, height=height * scale))

    paragraph("SAR Slick Lab", "Title")
    paragraph("AI II  /  Oil spill detection using semantic segmentation", "Heading2")
    if benchmark:
        paragraph(f"Status  /  Official held-out benchmark completed on {results['device']}.")
    elif synthetic:
        paragraph("Status  /  Pipeline verified on synthetic data. Official SAR benchmark pending.")
    else:
        paragraph("Status  /  Experimental methodology. Official SAR benchmark pending.")

    section("1. Objective and research background",
            "Segment sea, oil, look-alikes, ships and land in Synthetic Aperture Radar images. Oil slicks can dampen "
            "short surface waves, lowering radar backscatter. Other phenomena can also create dark regions. "
            "Krestenitis et al. (2019) introduced a labeled benchmark and found DeepLabV3+ strongest among the models "
            "they tested. The course dataset is expected to contain 1,002 training tiles and 110 official test tiles.")
    section("2. Data preparation and class imbalance",
            "Pair images and masks by filename stem and validate original dimensions and every RGB value before resizing. "
            "Accept integer labels 0 through 4 directly. Create a seeded 85/15 train-validation split from the "
            "official training folder. Apply bilinear image resize and nearest-neighbor mask resize. Use the same "
            "ImageNet RGB channel means and standard deviations for both models. Random horizontal flips and 90-degree "
            "rotations apply identically to training image and mask. Validation and test use no random transforms.")
    table([["ID", "Class", "RGB"], ["0", "Sea", "(0, 0, 0)"], ["1", "Oil spill", "(0, 255, 255)"],
           ["2", "Look-alike", "(255, 0, 0)"], ["3", "Ship", "(153, 76, 0)"], ["4", "Land", "(0, 153, 0)"]], [18, 55, 55])
    section("3. U-Net and DeepLabV3+",
            "U-Net starts from random weights. Four encoder blocks and max-pools precede a bottleneck. "
            "Bilinear decoder upsampling and skip concatenation restore fine spatial information. DeepLabV3+ "
            "starts its ResNet-18 encoder from torchvision ImageNet-1K V1 and fine-tunes all layers from epoch one. "
            "Atrous spatial pyramid pooling gathers multi-scale context, and a stride-four encoder skip restores "
            "local detail. Both heads return five logits per pixel on the input grid.")
    section("4. Controlled experiments",
            "Within each architecture, hold initialization, data split, augmentations, AdamW settings and epoch "
            "budget fixed. Compare plain cross-entropy with weighted cross-entropy plus soft Dice loss. Use capped "
            "inverse square-root frequency weights computed on resized training masks. Weighting increases minority "
            "class influence, and Dice directly rewards overlap. Rare-class weighting can also increase false alarms.")
    table([["Run", "Initialization", "Objective"], *EXPERIMENTS], [46, 51, 61])

    story.append(PageBreak())
    section("5. Monitoring and evaluation",
            "Select the earliest checkpoint with the highest validation mIoU. Record elapsed training time, loss, "
            "mIoU and class metrics each epoch. The official test set is loaded only after all four configurations "
            "are selected. IoU divides overlap by union, and Dice divides twice the overlap by the sum of both areas. "
            "Average IoU equally over classes with nonzero union. Pixel accuracy weights each pixel equally, so it "
            "can be high even when oil and ships are missed. Confusion rows represent ground truth.")
    if benchmark:
        audit = load(run / "audit.json")
        train_counts = audit["train"]["pixel_counts"]
        paragraph("Original training pixel proportions (Sea / Oil / Look-alike / Ship / Land)  " +
                  " / ".join(format_score(count / sum(train_counts)) for count in train_counts))
        paragraph("Training dimensions and class counts are available in audit.json.")
        table([["Experiment", "Epoch", "Val mIoU", "Test mIoU", "Accuracy", "Seconds"],
               *[[row["experiment"], row["best_epoch"], format_score(row["val_miou"]), format_score(row["miou"]),
                  format_score(row["accuracy"]), f"{row['seconds']:.1f}"] for row in results["experiments"]]],
              [44, 16, 25, 25, 25, 21])
        for metric in ("iou", "dice"):
            paragraph(f"Per-class test {metric.upper()}", "Heading3")
            table([["Experiment", "Sea", "Oil", "Look-alike", "Ship", "Land"],
                   *[[row["experiment"], *(format_score(x) for x in row[metric])] for row in results["experiments"]]],
                  [44, 21, 21, 27, 21, 21])
        ranked = max(results["experiments"], key=lambda row: row["miou"])
        paragraph(f"Highest held-out mIoU among the four fixed configurations was {ranked['experiment']} "
                  f"at {format_score(ranked['miou'])}. This is a descriptive test comparison.")
        for base, changed in zip(results["experiments"][::2], results["experiments"][1::2]):
            deltas = [changed["iou"][idx] - base["iou"][idx] if changed["iou"][idx] is not None
                      and base["iou"][idx] is not None else None for idx in (1, 3)]
            oil, ship = [f"{100 * x:+.2f} pp" if x is not None else "undefined" for x in deltas]
            paragraph(f"{changed['experiment']} vs. {base['experiment']}  |  oil IoU {oil}  |  ship IoU {ship}.")
    else:
        section("6. Results status",
                "Official mask audit, pretrained fine-tuning, held-out metrics, and SAR qualitative comparisons "
                "remain pending. The following verification uses generated geometric masks and noisy grayscale tiles."
                if synthetic else "Official dataset results and qualitative comparisons remain pending.")
        if synthetic:
            table([["Synthetic run", "Epoch", "Val mIoU", "Fixture mIoU", "Seconds"],
                   *[[row["experiment"], row["best_epoch"], format_score(row["val_miou"]),
                      format_score(row["miou"]), f"{row['seconds']:.1f}"] for row in results["experiments"]]],
                  [46, 20, 27, 32, 24])
            paragraph("The smoke test used one epoch per run, CPU, and random encoder weights for DeepLabV3+. "
                      "It exercises the implementation only. A one-point curve cannot establish convergence or overfitting.")

    section("7. Limitations and contribution statement",
            "Resizing can erase small ships and warp aspect ratios. A tile-level split may leave correlated scenes "
            "across training and validation when scene IDs are unavailable. A single seed does not estimate "
            "run-to-run variability. Hardware and library changes can affect reproducibility. "
            "Team names and individual contributions were not provided and require verification before submission.")
    section("References",
            "Krestenitis et al. (2019), Remote Sensing 11(15), 1762. DOI 10.3390/rs11151762. "
            "Kaggle course dataset: kaggle.com/datasets/nabilsherif/oil-spill. "
            "U-Net: arXiv:1505.04597. DeepLabV3+: arXiv:1802.02611.")
    if benchmark:
        story.append(PageBreak())
        paragraph("Dataset alignment preview", "Heading2")
        figure(run / "dataset-preview.png", 193)
        for row in results["experiments"]:
            story.append(PageBreak())
            name = row["experiment"]
            folder = run / name
            history = load(folder / "history.json")
            best = history[row["best_epoch"] - 1]
            last = history[-1]
            train_gain = last["train"]["miou"] - best["train"]["miou"]
            val_change = last["validation"]["miou"] - best["validation"]["miou"]
            paragraph(f"{name}  /  Learning curves", "Heading2")
            figure(folder / "history.png", 65)
            paragraph(f"From checkpoint epoch {row['best_epoch']} to final epoch {last['epoch']}, train mIoU changed "
                      f"{100 * train_gain:+.2f} pp and validation mIoU changed {100 * val_change:+.2f} pp.")
            if train_gain > .01 and val_change < -.01:
                paragraph("Training overlap improved after the selected epoch while validation declined, consistent "
                          "with a widening generalization gap. Inspect adjacent epochs for persistence.")
            elif last["epoch"] == row["best_epoch"]:
                paragraph("The highest validation mIoU was reached at the final epoch. Use the curves above to "
                          "inspect whether validation was still improving or had begun to flatten.")
            else:
                paragraph("The endpoint changes alone do not establish sustained divergence. Check the plotted "
                          "loss and mIoU trajectories for plateaus or oscillation.")
            figure(folder / "confusion.png", 105)
            matrix = row["confusion"]
            paragraph(f"Oil labeled as look-alike: {matrix[1][2]:,} pixels. Look-alike labeled as oil: "
                      f"{matrix[2][1]:,} pixels. Ships missed: {sum(matrix[3]) - matrix[3][3]:,} pixels.")
            story.append(PageBreak())
            paragraph(f"{name}  /  Held-out examples", "Heading2")
            figure(folder / "predictions.png", 177)
            for kind, example in row["examples"].items():
                paragraph(f"{kind}  |  {example['tile']}" +
                          (f"  |  {example['pixels']:,} pixels" if example["pixels"] is not None else ""))
    destination.parent.mkdir(parents=True, exist_ok=True)
    document = SimpleDocTemplate(str(destination), title="SAR Slick Lab", topMargin=14 * mm, bottomMargin=15 * mm)

    def footer(canvas, doc):
        canvas.setStrokeColor(TEAL)
        canvas.line(72, 38, doc.pagesize[0] - 72, 38)
        canvas.setFont("Helvetica", 8)
        canvas.setFillColor(INK)
        canvas.drawString(72, 24, "SAR SLICK LAB  /  AI II")
        canvas.drawRightString(doc.pagesize[0] - 72, 24, str(doc.page))

    document.build(story, onFirstPage=footer, onLaterPages=footer)


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--run", type=Path, default=Path("runs/final"))
    parser.add_argument("--pdf", type=Path, default=Path("SAR_Slick_Lab_Report.pdf"))
    args = parser.parse_args()
    build(args.run, args.pdf)



## Train all four configurations and compare their held-out predictions

In [ ]:
import subprocess
subprocess.run([sys.executable, "run.py", "--data", str(dataset), "--out", "final",
                "--device", "cuda", "--epochs", "16", "--size", "256", "--batch", "4"], check=True)

## View results and save the report
The report and evidence files appear under the notebook output directory.

In [ ]:
import importlib.util
import json
from IPython.display import Image, display

if importlib.util.find_spec("reportlab") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "reportlab>=4,<5"], check=True)
results = json.loads(Path("final/results.json").read_text())
print(json.dumps(results, indent=2))
display(Image(filename="final/dataset-preview.png"))
for row in results["experiments"]:
    print(row["experiment"])
    for filename in ["history.png", "confusion.png", "predictions.png"]:
        display(Image(filename=str(Path("final") / row["experiment"] / filename)))
subprocess.run([sys.executable, "report.py", "--run", "final",
                "--pdf", "SAR_Slick_Lab_Report.pdf"], check=True)